# CORE - A Cell-Level Coarse-to-Fine Image Registration Engine for Multi-stain Image Alignment

This notebook demonstrates the coarse level Whole Slide Image (WSI) registration using CORE method.

## Overview
- **Coarse Registration**: Initial coarse registration using CORE

> **Note:** This notebook has been patched to run standalone in Google Colab and to fix a few bugs present in the original version (a syntax error in the mask-visualization cell, and dependency/path setup). See the setup cell below.

## 1. Setup and Imports

In [3]:
# ============================================================
# Colab standalone setup
# ============================================================
import os, sys, subprocess, requests
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = "/content/CORE"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "https://github.com/eshasadia/test.git", REPO_DIR], check=True)
    os.chdir(REPO_DIR)

    %pip install -q "numpy<2.0" "scipy<1.14" "opencv-python-headless<4.10" \
        "scikit-image<0.23" "Pillow<11" "SimpleITK>=2.3.1,<2.4" "scikit-learn<1.4" \
        "pandas>=2.1,<2.2" "matplotlib<3.9" torch torchvision \
        "tiatoolbox==1.6" pycpd bokeh ipywidgets pillow-heif "av==12.0.0" openai

    %pip install -q vision-agent --no-deps
    %pip install -q --force-reinstall --no-deps "numpy<2.0"

    project_root = REPO_DIR
else:
    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"✅ Project root on sys.path: {project_root}")

# Vision Agent API key
if "VISION_AGENT_API_KEY" not in os.environ:
    api_key = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            api_key = userdata.get("VISION_AGENT_API_KEY")
        except Exception:
            api_key = None
    if not api_key:
        import getpass
        api_key = getpass.getpass("Enter VISION_AGENT_API_KEY (leave blank to skip): ")
    if api_key:
        os.environ["VISION_AGENT_API_KEY"] = api_key

# Data paths
global_save_dir = Path("content")
global_save_dir.mkdir(parents=True, exist_ok=True)
SOURCE_WSI_PATH = global_save_dir / "source_wsi.tiff"
TARGET_WSI_PATH = global_save_dir / "target_wsi.tiff"

r = requests.get(
    "https://huggingface.co/datasets/TIACentre/TIAToolBox_Remote_Samples/resolve/main/testdata/registration/CRC/06-18270_5_A1MLH1_1.tif",
    timeout=120,
)
r.raise_for_status()
with TARGET_WSI_PATH.open("wb") as f:
    f.write(r.content)

r = requests.get(
    "https://huggingface.co/datasets/TIACentre/TIAToolBox_Remote_Samples/resolve/main/testdata/registration/CRC/06-18270_5_A1MSH2_1.tif",
    timeout=120,
)
r.raise_for_status()
with SOURCE_WSI_PATH.open("wb") as f:
    f.write(r.content)

FIXED_POINTS_PATH = ""
MOVING_POINTS_PATH = ""

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.6/51.6 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 113.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.1/35.1 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 112.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 129.6 MB/s eta 0:00:00
   ━━━━

In [ ]:
# Enable inline plotting and auto-reload for development
import sys, importlib, types
if "imp" not in sys.modules:
    fake = types.ModuleType("imp")
    fake.reload = importlib.reload
    sys.modules["imp"] = fake
%matplotlib inline
%load_ext autoreload
%autoreload 2

# Import all necessary modules
import SimpleITK as sitk
from core.utils.imports import *
from core.config import *
from core.preprocessing.preprocessing import *
from core.preprocessing.padding import *
from core.registration.registration import *
from core.registration.nonrigid import *
from core.evaluation.evaluation import *
from core.visualization.visualization import *
from core.preprocessing.nuclei_analysis import *

# Setup Bokeh for notebook output
setup_bokeh_notebook()

print("✅ All modules imported successfully!")
print(f"Source WSI: {SOURCE_WSI_PATH}")
print(f"Target WSI: {TARGET_WSI_PATH}")

## 2. Configuration Check

Verify that all file paths are correct and files exist.

## 3. Load and Preprocess WSI Images

In [ ]:
import os

# Check if files exist
files_to_check = [
    SOURCE_WSI_PATH,
    TARGET_WSI_PATH,
    FIXED_POINTS_PATH,
    MOVING_POINTS_PATH
]

print("File existence check:")
for file_path in files_to_check:
    exists = os.path.exists(file_path)
    status = "✅" if exists else "❌"
    print(f"{status} {file_path}")

# Display current parameters
print("\nCurrent Parameters:")
print(f"- Preprocessing Resolution: {PREPROCESSING_RESOLUTION}")
print(f"- Registration Resolution: {REGISTRATION_RESOLUTION}")
print(f"- Patch Size: {PATCH_SIZE}")
print(f"- Fixed Threshold: {FIXED_THRESHOLD}")
print(f"- Moving Threshold: {MOVING_THRESHOLD}")
print(f"- Min Nuclei Area: {MIN_NUCLEI_AREA}")

In [ ]:
# Load WSI images
print("Loading WSI images...")
source_wsi, target_wsi, source, target = load_wsi_images(
    SOURCE_WSI_PATH, TARGET_WSI_PATH, PREPROCESSING_RESOLUTION
)

print(f"\nLoaded images:")
print(f"Source shape: {source.shape}")
print(f"Target shape: {target.shape}")

In [ ]:
import matplotlib.cm as cm
# Preprocess images
print("Preprocessing images...")
source_prep,target_prep, padding_params=pad_images(source, target)
# Extract tissue masks
print("Extracting tissue masks...")
# Set artefacts to true if your slide contains control tissue as well
source_mask, target_mask = extract_tissue_masks(source_prep, target_prep, artefacts=False)
print("✅ Preprocessing completed!")

## 4. Visualize Original Images and Tissue Masks

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

# Set to 'mif' to colorize a multiplex-IF DAPI channel target, or leave as
# 'brightfield' (default) to just show the target image as-is.
# (Original notebook referenced an undefined `slide` variable and had a
# missing colon / `taget` typo here - both fixed below.)
slide = 'brightfield'

if slide == 'mif':
    # Create a blank RGB image
    colored = np.zeros((*target_prep.shape, 3), dtype=np.uint8)

    # Apply colormap only on the masked region
    masked_region = target_prep.copy()
    masked_region[target_mask == 0] = 0  # zero out regions outside the mask

    colored_masked = cv2.applyColorMap(masked_region, cv2.COLORMAP_JET)
    colored_masked = cv2.cvtColor(colored_masked, cv2.COLOR_BGR2RGB)
else:
    colored_masked = target_prep

# Display side by side
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

axes[0, 0].imshow(source_prep)
axes[0, 0].set_title('Source Image (Moving)')
axes[0, 0].axis('off')

axes[0, 1].imshow(colored_masked)
axes[0, 1].set_title('Target Image (Masked & Colored)')
axes[0, 1].axis('off')

axes[1, 0].imshow(source_mask, cmap='gray')
axes[1, 0].set_title('Source Tissue Mask')
axes[1, 0].axis('off')

axes[1, 1].imshow(target_mask, cmap='gray')
axes[1, 1].set_title('Target Tissue Mask')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()


## 5. Perform Rigid Registration

In [ ]:
# Perform rigid registration
print("Performing rigid registration...")
moving_img_transformed, final_transform = perform_rigid_registration(
    source_prep, target_prep, source_mask, target_mask
)
# Visualize transformed image
plt.figure(figsize=(12, 8))
plt.imshow(moving_img_transformed)
plt.title('Transformed Source Image')
plt.axis('off')
plt.show()


In [ ]:
r_x, r_y = util.matrix_df(source_prep,np.linalg.inv(final_transform))
rigid_field = np.stack(( r_x, r_y), axis=-1)
sitk_image = sitk.GetImageFromArray(rigid_field)
# sitk.WriteImage(sitk_image, './anhir_rigid.mha')

In [ ]:

visualize_overlays(target_prep,source_prep, moving_img_transformed)

## 6. Non Rigid Registration

In [ ]:
displacement_field,warped_source= elastic_image_registration(
   moving_img_transformed,target_prep)
print("non rigid displacement field",displacement_field.shape)
create_deform(source_prep, final_transform, displacement_field, output_path="")

## 7. Visualization

**Note on Colab:** `tiatoolbox visualize` starts a local Bokeh server on `localhost`, which a hosted Colab runtime cannot reach directly. To use it in Colab you'll need to tunnel the port (e.g. with `pyngrok`) or download the outputs and run this command locally instead. The static overlay plots earlier in this notebook (`visualize_overlays`, etc.) work fine inline in Colab without any tunneling.

In [ ]:
%%bash
tiatoolbox visualize --slides "path/to/source/slide/folder" --overlays "path/to/target/slide/and/deformation/field/folder"